# Gradient boosting

LightGBM with early stopping, then CatBoost, on all features (`add_features`). The number of trees is chosen on
valid, so valid scores are slightly optimistic; the final check is on test.

In [1]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score

from fraud.data import load_train
from fraud.features import add_features
from fraud.split import time_split

In [2]:
df = add_features(load_train())
train, valid, _ = time_split(df)

NON_FEATURES = ['isFraud', 'TransactionID', 'TransactionDT']
X_train, y_train = train.drop(columns=NON_FEATURES), train['isFraud']
X_valid, y_valid = valid.drop(columns=NON_FEATURES),valid['isFraud']

In [3]:
MODEL_PARAMS = dict(
    n_estimators=3000,
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.5,
    metric='average_precision',
    n_jobs=8,
    verbose=-1,
)

for seed in [42,1,2]:
    model = lgb.LGBMClassifier(**MODEL_PARAMS, random_state=seed)
    model.fit(X_train, y_train, eval_X=X_valid, eval_y=y_valid, callbacks=[lgb.early_stopping(100, verbose=False)])
    scores = model.predict_proba(X_valid)[:, 1]
    print(f"seed {seed:<3} trees {model.best_iteration_:<5} "
          f"ROC-AUC {roc_auc_score(y_valid, scores):.4f}   PR-AUC {average_precision_score(y_valid, scores):.4f}")

seed 42  trees 1893  ROC-AUC 0.9290   PR-AUC 0.6510
seed 1   trees 1390  ROC-AUC 0.9309   PR-AUC 0.6483
seed 2   trees 1195  ROC-AUC 0.9327   PR-AUC 0.6442


With early stopping on valid PR-AUC (patience 100, up to 3,000 trees), LightGBM picks 1,893, 1,390 and 1,195 trees
for seeds 42, 1, 2 and reaches PR-AUC 0.6510, 0.6483 and 0.6442 (mean 0.648, ROC-AUC 0.931), against 0.607 with
300 trees: 300 trees were too few. The best number of trees varies a lot between seeds while the score barely
does, so the valid curve is flat near the top.